# Imports

In [ ]:
# === Imports & setup (warning-safe) ===
import os, sys, io, re, warnings
import contextlib
from pathlib import Path

# Silence the deprecation warning BEFORE importing torch/EDBO+
warnings.filterwarnings("ignore", category=UserWarning, message=r"pkg_resources is deprecated.*")

import numpy as np
import pandas as pd

# make edboplus importable from repo root
sys.path.append(os.path.abspath(os.path.join('..','..')))

# Suppress during import (belt & suspenders)
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=UserWarning, message=r"pkg_resources is deprecated.*")
    from edbo.plus.optimizer_botorch import EDBOplus
    try:
        import torch
    except ImportError:
        torch = None


import warnings
try:
    from botorch.utils.warnings import InputDataWarning
except Exception:
    class InputDataWarning(UserWarning):  # fallback if symbol moves between versions
        pass

warnings.filterwarnings("ignore", category=InputDataWarning, module="botorch.models.utils")

import shutil
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import MinMaxScaler
from boruta import BorutaPy
import matplotlib.pyplot as plt

# reproducibility
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
np.random.seed(SEED)
if torch is not None:
    torch.manual_seed(SEED)

# nicer DataFrame display
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:.6g}")

from IPython.display import display

# Active Learning Loop

## Step 1: read in data (tested and untested reactions) and features

In [ ]:
# === Step 1: Ingest AL data, patch prior-round measurements, build TRAIN/POOL, and visualize ===
# User-editable:
INPUT_XLSX = "CCs_Acetals.xlsx"   # your input spreadsheet (Structure, DeltaDeltaG, descriptors...)
ROUND_ID   = 0                          # the round you're about to run (0,1,2,...) — include all measured from rounds < ROUND_ID

# ----------------------------
# 0) Load input (do NOT modify it)
# ----------------------------
try:
    df_all = pd.read_excel(INPUT_XLSX, engine="openpyxl")
except Exception:
    df_all = pd.read_excel(INPUT_XLSX)

# Clean accidental Excel columns
drop_unnamed = [c for c in df_all.columns if str(c).startswith("Unnamed:")]
if drop_unnamed:
    df_all = df_all.drop(columns=drop_unnamed)

# ----------------------------
# 1) Basic validation & normalization
# ----------------------------
REQUIRED_COLS = ["Structure", "DeltaDeltaG"]
missing = [c for c in REQUIRED_COLS if c not in df_all.columns]
if missing:
    raise ValueError(f"Missing required columns in '{INPUT_XLSX}': {missing}")

df_all["Structure"]   = df_all["Structure"].astype(str).str.strip()
df_all["DeltaDeltaG"] = pd.to_numeric(df_all["DeltaDeltaG"], errors="coerce")

# Duplicate check
dup_mask = df_all["Structure"].duplicated(keep=False)
if dup_mask.any():
    print(f"[Warning] Duplicate Structure IDs found in input: "
          f"{sorted(df_all.loc[dup_mask, 'Structure'].unique().tolist())}")

# ----------------------------
# 2) Identify feature columns & coerce to numeric
# ----------------------------
feature_cols = [c for c in df_all.columns if c not in REQUIRED_COLS]
if len(feature_cols) == 0:
    raise ValueError("No descriptor/feature columns found after 'Structure' and 'DeltaDeltaG'.")

df_feats = df_all[feature_cols].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)

# Drop fully non-informative features (all-NaN)
all_nan_cols = df_feats.columns[df_feats.isna().all()].tolist()
if all_nan_cols:
    print(f"[Info] Dropping {len(all_nan_cols)} all-NaN feature(s).")
    df_feats = df_feats.drop(columns=all_nan_cols)

# Recombine (now all numeric features)
numeric_feature_cols = df_feats.columns.tolist()
df_all = pd.concat([df_all[REQUIRED_COLS], df_feats], axis=1)

# ----------------------------
# 3) Patch with prior-round measured results (from Step 5 ledgers)
#    - Use tested_results_ledger.csv if present; else, aggregate round{r}_tested_results.csv (r < ROUND_ID)
#    - NEVER write back to Excel; only patch df_all in-memory for modeling
# ----------------------------
def _load_prior_measured(round_id: int) -> pd.DataFrame:
    """Load all measured results from rounds < round_id."""
    ledger_path = Path("tested_results_ledger.csv")
    rows = []
    if ledger_path.exists():
        tmp = pd.read_csv(ledger_path)
        cols = {c.lower(): c for c in tmp.columns}
        # standardize column access
        c_round = cols.get("round", "round")
        c_struct = cols.get("structure", "Structure")
        c_y = cols.get("deltadeltag_true", "DeltaDeltaG_true") if "deltadeltag_true" in cols else \
              (cols.get("deltadeltag", "DeltaDeltaG") if "deltadeltag" in cols else "DeltaDeltaG_true")
        tmp = tmp.rename(columns={c_struct:"Structure", c_round:"round", c_y:"DeltaDeltaG_true"})
        tmp["Structure"] = tmp["Structure"].astype(str).str.strip()
        tmp["round"] = pd.to_numeric(tmp["round"], errors="coerce")
        tmp["DeltaDeltaG_true"] = pd.to_numeric(tmp["DeltaDeltaG_true"], errors="coerce")
        # keep r < round_id (strictly prior rounds)
        tmp = tmp[tmp["round"] < round_id]
        rows.append(tmp[["round","Structure","DeltaDeltaG_true"]])
    else:
        # fall back to per-round files if cumulative ledger not present
        for p in Path(".").glob("round*_tested_results.csv"):
            m = re.match(r"round(\d+)_tested_results\.csv$", p.name)
            if not m:
                continue
            r = int(m.group(1))
            if r >= round_id:
                continue
            tmp = pd.read_csv(p)
            tmp["Structure"] = tmp["Structure"].astype(str).str.strip()
            tmp["DeltaDeltaG_true"] = pd.to_numeric(tmp.get("DeltaDeltaG_true"), errors="coerce")
            tmp["round"] = r
            rows.append(tmp[["round","Structure","DeltaDeltaG_true"]])
    if not rows:
        return pd.DataFrame(columns=["round","Structure","DeltaDeltaG_true"])
    prior = pd.concat(rows, ignore_index=True)
    # if duplicates across files/rounds, keep the latest (highest round index)
    prior = prior.sort_values(["Structure","round"]).drop_duplicates("Structure", keep="last")
    # drop NaN y
    prior = prior.dropna(subset=["DeltaDeltaG_true"]).reset_index(drop=True)
    return prior

prior_meas = _load_prior_measured(ROUND_ID)

# Patch df_all in-memory with prior_meas where Excel had NaN y
df_all = df_all.copy()
df_all["_from_ledger"] = False  # track provenance for histogram overlay
if not prior_meas.empty:
    # join to get ledger y for matching structures
    df_all = df_all.merge(prior_meas[["Structure","DeltaDeltaG_true"]], on="Structure", how="left")
    # conflicts (Excel already has y and differs from ledger)
    both_present = df_all["DeltaDeltaG"].notna() & df_all["DeltaDeltaG_true"].notna()
    if both_present.any():
        diffs = (df_all.loc[both_present, "DeltaDeltaG"] - df_all.loc[both_present, "DeltaDeltaG_true"]).abs()
        n_conflict = int((diffs > 1e-12).sum())
        if n_conflict > 0:
            print(f"[Warning] {n_conflict} structure(s) have conflicting ΔΔG between Excel and ledger; "
                  "keeping Excel value and ignoring ledger for those.")
        # prefer Excel values whenever present; only use ledger where Excel is NaN
        df_all.loc[both_present, "DeltaDeltaG_true"] = np.nan

    # now fill NaN Excel y with ledger y and flag as from_ledger
    fill_mask = df_all["DeltaDeltaG"].isna() & df_all["DeltaDeltaG_true"].notna()
    df_all.loc[fill_mask, "DeltaDeltaG"] = df_all.loc[fill_mask, "DeltaDeltaG_true"]
    df_all.loc[fill_mask, "_from_ledger"] = True
    df_all = df_all.drop(columns=["DeltaDeltaG_true"])

# Any ledger structures not found in Excel?
if not prior_meas.empty:
    known_ids = set(df_all["Structure"])
    missing_ids = sorted(set(prior_meas["Structure"]) - known_ids)
    if missing_ids:
        print(f"[Warning] {len(missing_ids)} measured structure(s) in ledger not found in '{INPUT_XLSX}': {missing_ids[:8]}{'...' if len(missing_ids)>8 else ''}")

# ----------------------------
# 4) Split into TRAIN (measured) vs POOL/UNTESTED (unmeasured)
# ----------------------------
tested_mask = df_all["DeltaDeltaG"].notna()
df_train    = df_all.loc[tested_mask].reset_index(drop=True)     # measured → usable for training this round
df_pool     = df_all.loc[~tested_mask].reset_index(drop=True)    # unmeasured → search space for Step 3
df_untested = df_pool.copy()                                     # alias for downstream compatibility

# For downstream compatibility with earlier steps
df_train["Class"] = "train"
df_pool["Class"]  = "acetals"
df_all_with_class = pd.concat([df_train, df_pool], ignore_index=True)

# Handy matrices/arrays for modeling steps
X_train_full = df_train[numeric_feature_cols].copy()
y_train      = df_train["DeltaDeltaG"].astype(float).to_numpy()
X_pool_full  = df_pool[numeric_feature_cols].copy()

# Convenience references
columns_all_features = numeric_feature_cols
structures_train     = df_train["Structure"].tolist()
structures_pool      = df_pool["Structure"].tolist()
N_features_total     = len(feature_cols)
N_features_numeric   = len(numeric_feature_cols)

# ----------------------------
# 5) Reporting
# ----------------------------
n_from_ledger = int(df_train["_from_ledger"].sum()) if "_from_ledger" in df_train.columns else 0
print("=== Step 1 — Active-Learning Ingest (with prior-round patching) ===")
print(f"Round: {ROUND_ID}")
print(f"File : {INPUT_XLSX}")
print(f"Total reactions            : {len(df_all)}")
print(f"Measured in TRAIN (this rd): {len(df_train)}  (of which {n_from_ledger} patched from prior rounds)")
print(f"Untested (pool)            : {len(df_pool)}")
print(f"Total descriptors          : {N_features_total}")
print(f"Numeric descriptors        : {N_features_numeric}")
if all_nan_cols:
    print(f" - Dropped all-NaN features: {len(all_nan_cols)}")

# Missingness snapshot across numeric features (fraction of NaNs)
missing_frac = df_all[numeric_feature_cols].isna().mean()
n_feats_with_na = int((missing_frac > 0).sum())
print(f"Features with any missing values: {n_feats_with_na} / {N_features_numeric}")
if n_feats_with_na:
    top_missing = missing_frac.sort_values(ascending=False).head(10)
    print("Top-10 features by missing fraction:")
    for k, v in top_missing.items():
        print(f"  {k}: {v:.1%}")

# ----------------------------
# 6) Persist lightweight manifests (optional)
# ----------------------------
pd.DataFrame({"feature": numeric_feature_cols}).to_csv(f"round{ROUND_ID}_feature_names.csv", index=False)
df_train[["Structure", "DeltaDeltaG", "_from_ledger"]].to_csv(f"round{ROUND_ID}_train_manifest.csv", index=False)
df_pool[["Structure"]].to_csv(f"round{ROUND_ID}_pool_manifest.csv", index=False)
df_all_with_class.to_csv(f"round{ROUND_ID}_all_with_class.csv", index=False)

# ----------------------------
# 7) Visuals: histogram of target — overlay prior-round measurements
# ----------------------------
y_all_measured = df_train["DeltaDeltaG"].to_numpy()
y_from_ledger  = df_train.loc[df_train["_from_ledger"], "DeltaDeltaG"].to_numpy() if "_from_ledger" in df_train.columns else np.array([])

if y_all_measured.size > 0:
    plt.figure(figsize=(6.0, 4.2))
    # All measured (light)
    plt.hist(y_all_measured, bins=15, alpha=0.65, label="All measured (this round TRAIN)")
    # Overlay: those patched from prior rounds (darker)
    if y_from_ledger.size > 0:
        plt.hist(y_from_ledger, bins=20, alpha=0.85, label="Newly measured from prior rounds")
    else:
        print("[Info] No prior-round measured ΔΔG‡ found to highlight in histogram overlay.")
    plt.xlabel("ΔΔG‡ (kcal/mol)")
    plt.ylabel("Frequency")
    plt.title("Distribution of measured ΔΔG‡")
    plt.grid(True, linestyle="--", alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("[Info] No measured ΔΔG‡ values found yet — histogram skipped.")

# ----------------------------
# 8) Display the full data table (Structure, DeltaDeltaG, and all features)
# ----------------------------
print("\nFull dataset preview (all columns):")
display(df_all)

## Step 2: Feature curation

In [ ]:
# === Step 2: Feature curation with Boruta + target-aware collinearity pruning (loop-safe) ===
# User-editable:
ROUND_ID          = 0         # active-learning round you’re running now
CORR_THRESH       = 0.90      # drop features with |ρ| > CORR_THRESH against already-kept features
BORUTA_PERC       = 75        # Boruta: percentile threshold for shadow features (e.g., 60 or 75)
BORUTA_MAX_ITER   = 100        # Boruta: max iterations (e.g., 70–100)
RF_MAX_DEPTH      = 5         # RandomForest depth used inside Boruta
SEED              = 42 if "SEED" not in globals() else SEED

# ---------------------------------------------------------------------
# 0) Preconditions / inputs from Step 1
# ---------------------------------------------------------------------
assert "df_train" in globals(), "df_train not found. Please run Step 1 first."
assert {"Structure","DeltaDeltaG"}.issubset(df_train.columns), "df_train must contain 'Structure' and 'DeltaDeltaG'."

# Build full feature set from df_train (everything except id/target/provenance)
non_feature_cols = {"Structure", "DeltaDeltaG", "Class", "_from_ledger"}
full_feature_cols = [c for c in df_train.columns if c not in non_feature_cols]

# Coerce features to numeric and keep column names aligned
X_train_full = df_train[full_feature_cols].apply(pd.to_numeric, errors="coerce").copy()
y_train      = pd.to_numeric(df_train["DeltaDeltaG"], errors="coerce").to_numpy()

if len(df_train) == 0 or np.isnan(y_train).all():
    raise ValueError(f"[Round {ROUND_ID}] No measured targets available to fit Boruta. "
                     "Ensure Step 1 patched prior rounds or provide initial measurements.")
if len(df_train) < 4:
    print(f"[Warning][Round {ROUND_ID}] TRAIN is very small (n={len(df_train)}). "
          "Boruta/collinearity pruning may be unstable.")

# ---------------------------------------------------------------------
# 1) Boruta feature selection (on TRAIN ONLY) — median-impute X (Boruta requires no NaNs)
# ---------------------------------------------------------------------
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from boruta import BorutaPy
import numpy as np, pandas as pd

imputer_boruta = SimpleImputer(strategy="median")
X_train_imp    = imputer_boruta.fit_transform(X_train_full.values)
feature_names  = X_train_full.columns.tolist()

rf = RandomForestRegressor(
    n_jobs=-1,
    max_depth=RF_MAX_DEPTH,
    random_state=SEED
)
feat_selector = BorutaPy(
    estimator=rf,
    n_estimators='auto',
    verbose=2,
    random_state=SEED,
    perc=BORUTA_PERC,
    max_iter=BORUTA_MAX_ITER
)

print(f"[Round {ROUND_ID}] Fitting Boruta on TRAIN (n={len(df_train)}, features={len(feature_names)}) "
      f"with perc={BORUTA_PERC}, max_iter={BORUTA_MAX_ITER}...")
feat_selector.fit(X_train_imp, y_train)

support_strong = feat_selector.support_
support_weak   = feat_selector.support_weak_
ranking        = feat_selector.ranking_

confirmed_feats = [feature_names[i] for i, keep in enumerate(support_strong) if keep]
tentative_feats = [feature_names[i] for i, keep in enumerate(support_weak)   if keep]
selected_features = confirmed_feats + tentative_feats

# Fallback if Boruta keeps nothing
if len(selected_features) == 0:
    print(f"[Warning][Round {ROUND_ID}] Boruta selected 0 features — falling back to top-10 RF importances.")
    rf_fallback = RandomForestRegressor(n_estimators=200, max_depth=RF_MAX_DEPTH, n_jobs=-1, random_state=SEED)
    rf_fallback.fit(X_train_imp, y_train)
    importances = rf_fallback.feature_importances_
    order = np.argsort(importances)[::-1]
    topk = min(10, len(feature_names))
    selected_features = [feature_names[i] for i in order[:topk]]
    confirmed_feats = selected_features
    tentative_feats = []
    ranking = np.ones(len(feature_names), dtype=int)

# Save Boruta artifacts
boruta_summary = pd.DataFrame({
    "feature": feature_names,
    "decision": [
        "confirmed" if f in confirmed_feats else ("tentative" if f in tentative_feats else "rejected")
        for f in feature_names
    ],
    "rank": ranking
}).sort_values(["decision", "rank", "feature"], ascending=[True, True, True])

boruta_summary.to_csv(f"boruta_round{ROUND_ID}_feature_ranking.csv", index=False)
pd.Series(selected_features, name="selected_feature").to_csv(
    f"boruta_round{ROUND_ID}_selected_features.txt", index=False
)

print(f"[Round {ROUND_ID}] Boruta → confirmed: {len(confirmed_feats)} | "
      f"tentative: {len(tentative_feats)} | total kept: {len(selected_features)}")
print(f"Saved: boruta_round{ROUND_ID}_feature_ranking.csv, boruta_round{ROUND_ID}_selected_features.txt")

# ---------------------------------------------------------------------
# 2) Target-aware collinearity pruning on TRAIN (using Boruta-kept features)
#    - keep features most associated with y; drop highly collinear ones
# ---------------------------------------------------------------------
df_train_sel = df_train[selected_features].apply(pd.to_numeric, errors="coerce").copy()

# Drop all-NaN Boruta features (edge case)
all_nan_cols = [c for c in df_train_sel.columns if df_train_sel[c].isna().all()]
if all_nan_cols:
    print(f"[Round {ROUND_ID}] Dropping {len(all_nan_cols)} all-NaN feature(s) before corr check.")
    df_train_sel = df_train_sel.drop(columns=all_nan_cols)

# Median-impute for stable correlations
imputer_corr = SimpleImputer(strategy="median")
X_train_imp_df = pd.DataFrame(
    imputer_corr.fit_transform(df_train_sel),
    index=df_train_sel.index,
    columns=df_train_sel.columns
)

# Drop constants
constant_feats = [c for c in X_train_imp_df.columns if X_train_imp_df[c].nunique(dropna=True) <= 1]
X_noconst = X_train_imp_df.drop(columns=constant_feats) if constant_feats else X_train_imp_df.copy()

# Greedy pruning: sort by |corr(feature, y)| (tie-break by Boruta rank then feature name)
if X_noconst.shape[1] > 0:
    y_series = pd.Series(y_train, index=X_noconst.index, name="DeltaDeltaG")
    abs_r_to_y = X_noconst.corrwith(y_series).abs().fillna(0.0)

    # Use Boruta rank as tie-breaker where available
    rank_map = boruta_summary.set_index("feature")["rank"].to_dict()
    ordered_feats = sorted(
        X_noconst.columns,
        key=lambda f: (-abs_r_to_y.get(f, 0.0), rank_map.get(f, np.inf), f)
    )

    corr_abs = X_noconst.corr().abs()
    kept, dropped_corr = [], []
    for f in ordered_feats:
        if any(corr_abs.loc[f, k] > CORR_THRESH for k in kept):
            dropped_corr.append(f)
        else:
            kept.append(f)
    curated_features = kept
else:
    abs_r_to_y = pd.Series(dtype=float)
    dropped_corr = []
    curated_features = []

# Fallback if everything pruned
if len(curated_features) == 0:
    print(f"[Warning][Round {ROUND_ID}] All features pruned; falling back to top-10 by |corr(feature, y)|.")
    if X_noconst.shape[1] > 0:
        topk = min(10, X_noconst.shape[1])
        curated_features = (
            abs_r_to_y.sort_values(ascending=False).head(topk).index.tolist()
            if not abs_r_to_y.empty else list(X_noconst.columns)[:topk]
        )
    dropped_corr = []

# ---------------------------------------------------------------------
# 3) Save curation artifacts + expose for Step 3
# ---------------------------------------------------------------------
pd.Series(curated_features, name="curated_feature").to_csv(f"round{ROUND_ID}_curated_features.txt", index=False)
pd.DataFrame({"feature": constant_feats}).to_csv(f"round{ROUND_ID}_constants_removed.csv", index=False)
pd.DataFrame({
    "feature": dropped_corr,
    "abs_r_to_y": [abs_r_to_y.get(f, np.nan) for f in dropped_corr] if len(dropped_corr) else []
}).to_csv(f"round{ROUND_ID}_correlated_removed.csv", index=False)
if isinstance(abs_r_to_y, pd.Series) and not abs_r_to_y.empty:
    abs_r_to_y.rename("abs_r_to_y").to_csv(f"round{ROUND_ID}_feature_target_corr.csv", header=True)

# Expose objects for Step 3
feature_list_cur = curated_features
boruta_selected_features_cur = selected_features  # for reference in logs

# Optional: missingness snapshot on curated TRAIN matrix (pre-impute)
df_train_cur_raw = df_train[feature_list_cur].apply(pd.to_numeric, errors="coerce")
df_train_cur_raw.isna().mean().rename("missing_frac").to_csv(f"round{ROUND_ID}_curated_missingness.csv")

# ---------------------------------------------------------------------
# 4) Report
# ---------------------------------------------------------------------
print("\n=== Round {} — Feature curation summary ===".format(ROUND_ID))
print(f"TRAIN rows: {len(df_train)}")
print(f"Full features considered: {len(full_feature_cols)}")
print(f"Boruta kept: {len(selected_features)} (confirmed={len(confirmed_feats)}, tentative={len(tentative_feats)})")
print(f"Removed constants: {len(constant_feats)}")
print(f"Removed for |ρ| > {CORR_THRESH}: {len(dropped_corr)}")
print(f"Final curated features for Round {ROUND_ID}: {len(feature_list_cur)}")

if isinstance(abs_r_to_y, pd.Series) and not abs_r_to_y.empty:
    print("\nTop-10 features by |corr(feature, ΔΔG)| on TRAIN this round:")
    print(abs_r_to_y.sort_values(ascending=False).head(10))

## Step 3: Train GPR model and predict on untested reactions

In [ ]:
# === Step 3: Train GP (EDBO+), predict UNTESTED, SHAP on EDBO GP ===

# ---------------------------
# User-editable
# ---------------------------
ROUND_ID = 0
SEED     = 42 if "SEED" not in globals() else SEED

# --- SHAP controls ---
DO_SHAP     = True

MAX_DISPLAY = 10            # top-N features to show in plots
N_BG        = 128           # background rows for SHAP (speed vs variance)
N_EVAL_ROWS = 275           # points used to compute SHAP (speed vs variance)
SHAP_SEED   = 42
PLOT_TYPES  = ["bar", "beeswarm"]  # any subset of {"bar","beeswarm"}

# --- Important featurs (mean|SHAP| is in kcal/mol here) ---
SHAP_ABS_THR   = 0.10       # kcal/mol (e.g., 0.05 lenient; 0.10 stricter)
SHAP_COVERAGE  = 0.90       # fraction of total mean|SHAP| mass to cover (e.g., 0.90 or 0.95)

# Output Excel with (Structure, DeltaDeltaG, important features) for ALL datapoints
WRITE_IMPORTANT_XLSX = True

# ---------------------------
# Imports + deterministic seeds
# ---------------------------
import os, sys, io, re, math, random, warnings, contextlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score

# Ensure repo root is importable (matches your Imports cell)
repo_root = os.path.abspath(os.path.join("..", ".."))
if repo_root not in sys.path:
    sys.path.append(repo_root)

# determinism (model training + edbo run)
random.seed(SEED)
np.random.seed(SEED)

import torch
torch.manual_seed(SEED)
try:
    torch.use_deterministic_algorithms(True)
except Exception:
    pass

# ---- EDBO+ imports (YOUR package lives under edbo.plus.*) ----
try:
    from edbo.plus.optimizer_botorch import EDBOplus
    from edbo.plus.model import build_and_optimize_model
    from edbo.plus.utils import EDBOStandardScaler
except ModuleNotFoundError:
    # fallback if someone installed it as edboplus (rare in your setup)
    from edboplus.optimizer_botorch import EDBOplus
    from edboplus.model import build_and_optimize_model
    from edboplus.utils import EDBOStandardScaler

from botorch.models import SingleTaskGP

# ---------------------------
# 0) Compatibility & sanity checks (uses results from Steps 1–2)
# ---------------------------
# df_train (measured)
if "df_train" not in globals():
    if "df_measured" in globals():
        df_train = df_measured.copy()
    else:
        raise AssertionError("df_train not found. Run Step 1.")

# df_untested (AL pool)
if "df_untested" not in globals():
    if "df_pool" in globals():
        df_untested = df_pool.copy()
    elif "df_all" in globals():
        assert "DeltaDeltaG" in df_all.columns, "df_all missing 'DeltaDeltaG'. Run Step 1."
        df_untested = df_all[df_all["DeltaDeltaG"].isna()].copy()
    else:
        raise AssertionError("df_untested not found. Run Step 1.")

# df_all (for final XLSX output)
assert "df_all" in globals(), "df_all not found. Run Step 1 (it builds df_all with patched ΔΔG‡)."
df_all_local = df_all.copy()

# feature list (curated)
if "feature_list_cur" not in globals():
    if "curated_features" in globals():
        feature_list_cur = list(curated_features)
    elif "selected_features" in globals():
        feature_list_cur = list(selected_features)
    else:
        raise AssertionError("feature_list_cur not found. Run Step 2.")

feat_cols = list(feature_list_cur)
assert len(feat_cols) > 0, "feature_list_cur is empty. Check Step 2."

# normalize keys
for _df in [df_train, df_untested, df_all_local]:
    _df["Structure"] = _df["Structure"].astype(str).str.strip()

# Ensure we have measured targets to train on
if df_train["DeltaDeltaG"].dropna().empty:
    raise ValueError(
        f"[Round {ROUND_ID}] No measured ΔΔG‡ values available for training. "
        "Make sure Step 1 patched previous rounds' results or input initial measurements."
    )

# ---------------------------
# 1) Helpers
# ---------------------------
def _clean_and_impute_with_train_means(df_train, df_any, feat_cols):
    """Coerce to numeric, replace inf, and fill NaNs with TRAIN column means."""
    Xtr  = df_train[feat_cols].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    Xany = df_any[feat_cols].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    means = Xtr.mean(axis=0)
    Xtr  = Xtr.fillna(means)
    Xany = Xany.fillna(means)
    return Xtr, Xany

def _detect_mean_std_cols(df):
    cols = df.columns.tolist()
    preferred = [
        ("DeltaDeltaG_predicted_mean", "DeltaDeltaG_predicted_variance"),
        ("DeltaDeltaG_mean", "DeltaDeltaG_std"),
        ("DeltaDeltaG_mu", "DeltaDeltaG_sigma"),
        ("posterior_mean_DeltaDeltaG", "posterior_std_DeltaDeltaG"),
        ("DeltaDeltaG_pred", "DeltaDeltaG_uncertainty"),
        ("mean_0", "std_0"), ("mu_0", "sigma_0"),
        ("pred_mean", "pred_std"), ("mean", "std"),
    ]
    for m, s in preferred:
        if m in cols and s in cols:
            return m, s
    mean_like = [c for c in cols if re.search(r"(mean|mu|pred)", c, re.I)]
    std_like  = [c for c in cols if re.search(r"(std|sigma|uncert|var)", c, re.I)]
    if mean_like and std_like:
        return mean_like[0], std_like[0]
    raise ValueError("Could not detect prediction mean/std columns in the EDBO+ predictions file.")

def _detect_ei_col(df):
    cols = df.columns.tolist()
    preferred = [
        "expected_improvement", "Expected_Improvement", "EI", "ei",
        "expected_improvement_0", "EI_0", "qEI", "qEI_0",
        "acqf", "acquisition_value", "acquisition_function", "acq_value",
    ]
    for c in preferred:
        if c in cols:
            return c
    regex_hits = [c for c in cols if re.search(r"(expected.*improv)|(^|_)ei(_|$)|acq", c, re.I)]
    return regex_hits[0] if regex_hits else None

def _re_emit_filtered(text: str):
    if not text:
        return
    for line in text.splitlines():
        s = line.strip()
        if s.startswith("The following features will be used"): continue
        if s.startswith("This run will optimize for the following objectives"): continue
        print(line)

def _prior_pool_best(round_id: int) -> float:
    """Max ΔΔG among pool items measured in prior rounds (ledger preferred)."""
    ledger = Path("tested_results_ledger.csv")
    if ledger.exists():
        try:
            led = pd.read_csv(ledger)
            led.columns = [str(c).strip() for c in led.columns]
            if "round" in led.columns and "DeltaDeltaG_true" in led.columns:
                led_f = led[pd.to_numeric(led["round"], errors="coerce") < round_id]
                y = pd.to_numeric(led_f["DeltaDeltaG_true"], errors="coerce")
                y = y[np.isfinite(y)]
                if len(y):
                    return float(y.max())
        except Exception:
            pass

    prev_selected = set()
    for p in Path(".").glob("round*_batch*_selection.csv"):
        m = re.match(r"round(\d+)_batch.*_selection\.csv$", p.name)
        if m and int(m.group(1)) < round_id:
            try:
                tmp = pd.read_csv(p)
                if "Structure" in tmp.columns:
                    prev_selected.update(tmp["Structure"].astype(str).str.strip().tolist())
            except Exception:
                pass
    if not prev_selected:
        return float("nan")

    df_train_keys = df_train[["Structure", "DeltaDeltaG"]].copy()
    df_train_keys["Structure"] = df_train_keys["Structure"].astype(str).str.strip()
    df_pool_measured = df_train_keys[df_train_keys["Structure"].isin(prev_selected)].copy()
    if df_pool_measured.empty:
        return float("nan")

    y = pd.to_numeric(df_pool_measured["DeltaDeltaG"], errors="coerce")
    y = y[np.isfinite(y)]
    return float(y.max()) if len(y) else float("nan")

# ---------------------------
# 2) Prepare EDBO+ input (Train observed + UNTESTED as PENDING)
# ---------------------------
Xtr_imp, Xun_imp = _clean_and_impute_with_train_means(df_train, df_untested, feat_cols)

df_train_in = pd.concat(
    [df_train[["Structure", "DeltaDeltaG"]].reset_index(drop=True),
     Xtr_imp.reset_index(drop=True)],
    axis=1
)
df_train_in.insert(1, "Class", "train")

df_untested_in = pd.concat(
    [df_untested[["Structure"]].reset_index(drop=True),
     Xun_imp.reset_index(drop=True)],
    axis=1
)
df_untested_in.insert(1, "Class", "pool")
df_untested_in["DeltaDeltaG"] = "PENDING"

cols_out = ["Structure", "Class", "DeltaDeltaG"] + feat_cols
df_edbo_in = pd.concat([df_train_in[cols_out], df_untested_in[cols_out]], ignore_index=True)

infile = f"edbo_gpr_round{ROUND_ID}.csv"
df_edbo_in.to_csv(infile, index=False)
print(f"[EDBO+] Input written: {infile}")
print(f"Rows total: {len(df_edbo_in)} | Train (measured): {len(df_train_in)} | Untested: {len(df_untested_in)} | Features used: {len(feat_cols)}")
print("Running EDBO+ with MinMaxScaler on features (suppressing verbose feature lists)...")

# ---------------------------
# 3) Run EDBO+ quietly + MinMaxScaler + suppress BoTorch warnings
# ---------------------------
buf_out, buf_err = io.StringIO(), io.StringIO()

try:
    from botorch.exceptions import InputDataWarning          # older BoTorch
except Exception:
    try:
        from botorch.exceptions.warnings import InputDataWarning  # newer BoTorch
    except Exception:
        InputDataWarning = UserWarning

with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore",
        message=r"Input data is not contained to the unit cube.*",
        category=UserWarning,
        module=r"botorch\.models\.utils"
    )
    warnings.filterwarnings(
        "ignore",
        message=r"Input data is not standardized.*",
        category=UserWarning,
        module=r"botorch\.models\.utils"
    )
    warnings.filterwarnings("ignore", category=InputDataWarning)

    with contextlib.redirect_stdout(buf_out), contextlib.redirect_stderr(buf_err):
        _ = EDBOplus().run(
            filename=infile,
            objectives=["DeltaDeltaG"],
            objective_mode=["max"],
            batch=4,
            columns_features=feat_cols,
            init_sampling_method="cvt",
            seed=SEED,
            scaler_features=MinMaxScaler(feature_range=(0.0, 1.0)),
            scaler_objectives=EDBOStandardScaler(),
        )

_re_emit_filtered(buf_out.getvalue())
_re_emit_filtered(buf_err.getvalue())

# ---------------------------
# 4) Load predictions (EDBO+ writes 'pred_<input-filename>') & detect columns
# ---------------------------
pred_path = Path(f"pred_{infile}")
if not pred_path.exists():
    alts = list(Path(".").glob("pred_*round*.csv")) + list(Path(".").glob("pred_*.csv"))
    alts = sorted(alts, key=lambda p: p.stat().st_mtime, reverse=True)
    if not alts:
        raise FileNotFoundError("Could not find an EDBO+ predictions CSV (pred_*).")
    pred_path = alts[0]

pred = pd.read_csv(pred_path)
print(f"[EDBO+] Predictions loaded: {pred_path.name} (rows={len(pred)})")

mean_col, unc_col = _detect_mean_std_cols(pred)
print(f"[EDBO+] Using prediction columns → mean: '{mean_col}', uncertainty/var: '{unc_col}'")

# ---------------------------
# 5) TRAIN metrics (sanity check on rows we sent as TRAIN)
# ---------------------------
pred_train = pred.merge(df_train_in[["Structure"]], on="Structure", how="inner")
if "DeltaDeltaG" not in pred_train.columns or pred_train["DeltaDeltaG"].isna().all():
    pred_train = pred_train.merge(
        df_train[["Structure", "DeltaDeltaG"]],
        on="Structure", how="left", suffixes=("", "_true")
    )

y_true = pd.to_numeric(pred_train["DeltaDeltaG"], errors="coerce").to_numpy()
y_hat  = pd.to_numeric(pred_train[mean_col], errors="coerce").to_numpy()
mask   = np.isfinite(y_true) & np.isfinite(y_hat)
y_true = y_true[mask]; y_hat = y_hat[mask]

mse  = mean_squared_error(y_true, y_hat) if len(y_true) else np.nan
rmse = float(np.sqrt(mse)) if np.isfinite(mse) else np.nan
mae  = float(np.mean(np.abs(y_true - y_hat))) if len(y_true) else np.nan
r2   = r2_score(y_true, y_hat) if (len(y_true) and len(np.unique(y_true)) > 1) else np.nan

print(f"\n=== GP (EDBO+) — Train performance (Round {ROUND_ID}) ===")
print(f"R²   : {r2: .4f}" if np.isfinite(r2) else "R²   :  n/a")
print(f"RMSE : {rmse: .4f} kcal/mol" if np.isfinite(rmse) else "RMSE :  n/a")
print(f"MAE  : {mae: .4f}  kcal/mol" if np.isfinite(mae)  else "MAE  :  n/a")

pd.DataFrame({"metric": ["R2", "RMSE", "MAE"], "value": [r2, rmse, mae]}).to_csv(
    f"round{ROUND_ID}_train_metrics.csv", index=False
)

# ---------------------------
# 6) UNTESTED predictions + σ + EI (EI vs prior-pool best; NaN at Round 0)
# ---------------------------
pred_untested = pred.merge(df_untested_in[["Structure"]], on="Structure", how="inner").copy()

mu = pd.to_numeric(pred_untested[mean_col], errors="coerce").to_numpy()
if any(k in unc_col.lower() for k in ["var", "variance"]):
    sigma = np.sqrt(np.clip(pd.to_numeric(pred_untested[unc_col], errors="coerce").to_numpy(), 0.0, None))
else:
    sigma = pd.to_numeric(pred_untested[unc_col], errors="coerce").to_numpy()

ei_edbo_col = _detect_ei_col(pred_untested)
ei_vs_full_train = (
    pd.to_numeric(pred_untested[ei_edbo_col], errors="coerce").to_numpy()
    if ei_edbo_col else np.full(len(pred_untested), np.nan)
)

f_best_pool = _prior_pool_best(ROUND_ID)
if np.isfinite(f_best_pool):
    print(f"\n[EI] Referencing improvement to prior measured pool best: f_best_pool = {f_best_pool:.6g} kcal/mol")
    eps = 1e-12
    sigma_safe = np.where(np.isfinite(sigma) & (sigma > eps), sigma, np.nan)
    z = (mu - f_best_pool) / sigma_safe
    try:
        from scipy.stats import norm
        Phi = norm.cdf(z); phi = norm.pdf(z)
    except Exception:
        erf_vec = np.vectorize(math.erf)
        Phi = 0.5 * (1.0 + erf_vec(z / np.sqrt(2.0)))
        phi = (1.0 / np.sqrt(2.0 * np.pi)) * np.exp(-0.5 * z * z)
    ei_vs_pool = np.where(
        np.isfinite(sigma_safe),
        (mu - f_best_pool) * Phi + sigma_safe * phi,
        np.maximum(0.0, mu - f_best_pool)
    )
else:
    print("\n[EI] No prior measured pool items found (e.g., Round 0) — EI left as NaN.")
    ei_vs_pool = np.full_like(mu, np.nan, dtype=float)

untested_out = pd.DataFrame({
    "Structure": pred_untested["Structure"].astype(str).values,
    "DeltaDeltaG_pred": mu,
    "DeltaDeltaG_uncertainty": sigma,       # σ (std), not variance
    "Expected_Improvement": ei_vs_pool,     # EI vs prior pool best
    "EI_vs_full_train": ei_vs_full_train,   # audit only
}).sort_values("DeltaDeltaG_pred", ascending=False).reset_index(drop=True)

untested_out_path = f"round{ROUND_ID}_acetals_predictions.csv"
full_pred_path    = f"round{ROUND_ID}_predictions_full.csv"
untested_out.to_csv(untested_out_path, index=False)
pred.to_csv(full_pred_path, index=False)

acetals_out = untested_out.copy()

print(f"\nSaved: round{ROUND_ID}_train_metrics.csv, {untested_out_path}, {full_pred_path}")
print(f"=== UNTESTED predictions (Round {ROUND_ID}, sorted by predicted ΔΔG‡ desc) ===")
display(untested_out.head(12))

# ---------------------------
# 7) Histogram of predicted ΔΔG‡ on UNTESTED
# ---------------------------
if not untested_out.empty:
    plt.figure(figsize=(5.2, 3.8))
    plt.hist(untested_out["DeltaDeltaG_pred"].astype(float), bins=20)
    plt.xlabel("Predicted ΔΔG‡ (kcal/mol)")
    plt.ylabel("Frequency")
    plt.title(f"Round {ROUND_ID} — Predicted ΔΔG‡ distribution (UNTESTED)")
    plt.grid(True, linestyle="--", alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("[Info] No untested candidates to plot.")

# ============================================================================
# 8) SHAP on recreated EDBO+ GP (same training code path) + verify vs pred_*
#    + choose "important" features via ABS+Coverage rule
#    + write an XLSX with Structure, DeltaDeltaG, important features (ALL rows, original order)
# ============================================================================
if DO_SHAP:
    import shap
    import matplotlib as mpl
    import matplotlib.gridspec as gridspec
    from matplotlib.colors import Normalize
    from matplotlib.cm import ScalarMappable

    random.seed(SHAP_SEED)
    np.random.seed(SHAP_SEED)
    torch.manual_seed(SHAP_SEED)
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass

    rng = np.random.default_rng(SHAP_SEED)

    # recreate same TRAIN matrices
    Xtr_np = df_train_in[feat_cols].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)
    y_raw  = df_train_in["DeltaDeltaG"].astype(float).to_numpy().reshape(-1, 1)

    scaler_x = MinMaxScaler(feature_range=(0.0, 1.0))
    Xtr_scaled = scaler_x.fit_transform(Xtr_np)

    scaler_y = EDBOStandardScaler()
    y_scaled = scaler_y.fit_transform(y_raw)

    tkwargs = {"dtype": torch.double, "device": torch.device("cpu")}
    train_x_t = torch.tensor(Xtr_scaled, **tkwargs)
    train_y_t = torch.tensor(y_scaled, **tkwargs)

    gp_opt, likelihood = build_and_optimize_model(train_x=train_x_t, train_y=train_y_t)
    surrogate_model = SingleTaskGP(
        train_X=train_x_t,
        train_Y=train_y_t,
        covar_module=gp_opt.covar_module,
        likelihood=likelihood
    ).to(**tkwargs)
    surrogate_model.eval()

    # verify vs pred_* mean
    X_all_np = pd.concat([df_train_in[feat_cols], df_untested_in[feat_cols]], ignore_index=True)\
                .apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)
    X_all_scaled = scaler_x.transform(X_all_np)
    X_all_t = torch.tensor(X_all_scaled, **tkwargs)

    with torch.no_grad():
        mu_scaled_all = surrogate_model.posterior(X_all_t).mean.detach().cpu().numpy()
        mu_all = scaler_y.inverse_transform(mu_scaled_all).reshape(-1)

    tmp = pd.DataFrame({
        "Structure": pd.concat([df_train_in["Structure"], df_untested_in["Structure"]], ignore_index=True).astype(str).values,
        "mu_recreated": mu_all,
    })
    tmp = tmp.merge(pred[["Structure", mean_col]].copy(), on="Structure", how="inner")
    tmp["mu_edbo"] = pd.to_numeric(tmp[mean_col], errors="coerce")
    tmp = tmp[np.isfinite(tmp["mu_recreated"]) & np.isfinite(tmp["mu_edbo"])]

    if len(tmp):
        max_abs = float(np.max(np.abs(tmp["mu_recreated"] - tmp["mu_edbo"])))
        mae_abs = float(np.mean(np.abs(tmp["mu_recreated"] - tmp["mu_edbo"])))
        print(f"\n[SHAP][VERIFY] Recreated GP vs pred_* mean:  MAE={mae_abs:.6g}  max|Δ|={max_abs:.6g}")
        if max_abs > 1e-3:
            print("[SHAP][VERIFY] WARNING: max|Δ| > 1e-3 → SHAP is on a close-but-not-identical GP.")
        else:
            print("[SHAP][VERIFY] OK: predictions match closely; SHAP is effectively on the EDBO surrogate.")
    else:
        print("\n[SHAP][VERIFY] Could not compare (no overlap / non-numeric). Proceeding anyway.")

    # background/eval from TRAIN only
    n = len(df_train_in)
    bg_idx   = rng.choice(n, size=min(N_BG, n), replace=False)
    eval_idx = rng.choice(n, size=min(N_EVAL_ROWS, n), replace=False)

    X_bg   = Xtr_scaled[bg_idx]
    X_eval = Xtr_scaled[eval_idx]
    S_eval = df_train_in["Structure"].to_numpy()[eval_idx]

    bg_df   = pd.DataFrame(X_bg,   columns=feat_cols)
    eval_df = pd.DataFrame(X_eval, columns=feat_cols)

    def predict_fn(Xdf: pd.DataFrame) -> np.ndarray:
        Xdf = Xdf.reindex(columns=feat_cols)
        X = torch.tensor(Xdf.to_numpy(), **tkwargs)
        with torch.no_grad():
            mu_scaled = surrogate_model.posterior(X).mean.detach().cpu().numpy()
        return scaler_y.inverse_transform(mu_scaled)  # (n,1) kcal/mol

    masker = shap.maskers.Independent(bg_df)
    try:
        explainer = shap.Explainer(predict_fn, masker, algorithm="permutation", seed=SHAP_SEED)
    except TypeError:
        np.random.seed(SHAP_SEED)
        explainer = shap.Explainer(predict_fn, masker, algorithm="permutation")

    sv = explainer(eval_df, max_evals=min(2*len(feat_cols)+1, 2*MAX_DISPLAY*len(eval_df)+1))

    vals = sv.values
    if vals.ndim == 3:
        vals = vals[:, 0, :]

    # mean|SHAP| in kcal/mol, stable sort
    mean_abs = pd.Series(np.abs(vals).mean(axis=0), index=feat_cols)
    mean_abs_sorted = mean_abs.sort_values(ascending=False, kind="mergesort")
    total_mass = float(mean_abs_sorted.sum()) if len(mean_abs_sorted) else 0.0

    # For plotting we still show top MAX_DISPLAY
    top_feats = mean_abs_sorted.index[:min(MAX_DISPLAY, len(mean_abs_sorted))].tolist()

    # ---------------------------
    # IMPORTANT FEATURES: ABS + COVERAGE rule
    # ---------------------------
    important_feats = []
    achieved = 0.0

    if total_mass > 0:
        running = 0.0
        # take only features above abs threshold, and add in rank order until coverage reached
        for f, v in mean_abs_sorted.items():
            if float(v) < float(SHAP_ABS_THR):
                continue
            important_feats.append(f)
            running += float(v)
            achieved = running / total_mass
            if achieved >= float(SHAP_COVERAGE):
                break

        if not important_feats:
            print(f"[SHAP][IMPORTANT] No features met abs threshold {SHAP_ABS_THR:.3g} kcal/mol; falling back to coverage-only.")
        if achieved < float(SHAP_COVERAGE):
            # fallback: coverage-only (top-k by mass)
            cum = mean_abs_sorted.cumsum() / total_mass
            k = int((cum < float(SHAP_COVERAGE)).sum()) + 1
            important_feats = mean_abs_sorted.index[:k].tolist()
            achieved = float(mean_abs_sorted.loc[important_feats].sum() / total_mass)
            print(f"[SHAP][IMPORTANT] WARNING: Could not reach {SHAP_COVERAGE:.0%} coverage using abs≥{SHAP_ABS_THR:.3g}. "
                  f"Using coverage-only top-{len(important_feats)} instead.")
    else:
        print("[SHAP][IMPORTANT] Total mean|SHAP| mass is zero/empty; cannot define important features.")

    print(f"\n[SHAP][IMPORTANT] Rule: abs≥{SHAP_ABS_THR:.3g} kcal/mol + cover≥{SHAP_COVERAGE:.0%} of total mean|SHAP|")
    print(f"[SHAP][IMPORTANT] Important feature count: {len(important_feats)} | Achieved coverage: {achieved:.1%}")
    if len(important_feats):
        print("[SHAP][IMPORTANT] Important features (ranked):")
        for i, f in enumerate(important_feats, 1):
            print(f"  {i:>2d}. {f}  (mean|SHAP|={mean_abs_sorted[f]:.6g})")

    out_dir = Path(f"round{ROUND_ID}_shap")
    out_dir.mkdir(exist_ok=True)

    # Save mean|SHAP| for all features + important list
    mean_abs_sorted.rename("mean_abs_shap").to_csv(out_dir / f"round{ROUND_ID}_mean_abs_shap.csv", header=True)
    pd.DataFrame({
        "feature": important_feats,
        "mean_abs_shap": [float(mean_abs_sorted[f]) for f in important_feats],
    }).to_csv(out_dir / f"round{ROUND_ID}_important_features_abs{SHAP_ABS_THR}_cov{int(SHAP_COVERAGE*100)}.csv", index=False)

    print(f"\n[SHAP] Saved mean |SHAP| → {out_dir}/round{ROUND_ID}_mean_abs_shap.csv")
    print(f"[SHAP] Saved important feature list → {out_dir}/round{ROUND_ID}_important_features_abs{SHAP_ABS_THR}_cov{int(SHAP_COVERAGE*100)}.csv")

    # ---------------------------
    # Write IMPORTANT-FEATURES XLSX (ALL datapoints; patched ΔΔG‡; original order)
    # ---------------------------
    if WRITE_IMPORTANT_XLSX:
        # ensure important feats are actually present in df_all
        missing_cols = [c for c in important_feats if c not in df_all_local.columns]
        if missing_cols:
            print(f"[SHAP][XLSX] WARNING: {len(missing_cols)} important feature(s) not found in df_all; they will be skipped.")
        important_in_all = [c for c in important_feats if c in df_all_local.columns]

        # preserve original order; if Step 1 ever provides a lock column, use it
        df_out_base = df_all_local.copy()
        if "_row_id" in df_out_base.columns:
            df_out_base = df_out_base.sort_values("_row_id", kind="mergesort").reset_index(drop=True)

        # build output with requested structure
        df_out = df_out_base[["Structure", "DeltaDeltaG"] + important_in_all].copy()

        # file name: mirrors input structure but filtered
        input_name = globals().get("INPUT_XLSX", None)
        stem = Path(str(input_name)).stem if input_name else f"round{ROUND_ID}"
        xlsx_path = out_dir / f"{stem}_Round{ROUND_ID}_ImportantFeatures.xlsx"

        df_out.to_excel(xlsx_path, index=False, engine="openpyxl")
        print(f"[SHAP][XLSX] Wrote filtered dataset → {xlsx_path}  (rows={len(df_out)}, features={len(important_in_all)})")

    # ---------------------------
    # long-form beeswarm WITHOUT jitter (top MAX_DISPLAY only)
    # ---------------------------
    long_rows = []
    row_centers = {f: (len(top_feats)-1 - i) for i, f in enumerate(top_feats)}  # most important at top
    for f in top_feats:
        j = feat_cols.index(f)
        shap_f = vals[:, j]
        x_fval = eval_df[f].to_numpy()
        y0 = row_centers[f]
        y_plot = np.full(len(eval_df), y0, dtype=float)  # NO jitter

        for i in range(len(eval_df)):
            long_rows.append({
                "Structure": S_eval[i],
                "feature": f,
                "row_index": int(i),
                "row_center": float(y0),
                "y_plot": float(y_plot[i]),
                "shap_value": float(shap_f[i]),
                "feature_value_scaled": float(x_fval[i]),
            })

    beeswarm_df = pd.DataFrame(long_rows)
    beeswarm_path = out_dir / f"round{ROUND_ID}_shap_beeswarm_top{len(top_feats)}.csv"
    beeswarm_df.to_csv(beeswarm_path, index=False)
    print(f"[SHAP] Saved beeswarm data → {beeswarm_path}")

    # Bar plot (top MAX_DISPLAY)
    if any(pt.lower() == "bar" for pt in PLOT_TYPES) and len(top_feats) > 0:
        bar_vals = mean_abs_sorted.loc[top_feats].iloc[::-1]
        fig, ax = plt.subplots(figsize=(6.4, 0.5*len(top_feats) + 1.8))
        ax.barh(np.arange(len(bar_vals)), bar_vals.values)
        ax.set_yticks(np.arange(len(bar_vals)))
        ax.set_yticklabels(bar_vals.index)
        ax.set_xlabel("mean |SHAP| (ΔΔG‡ contribution, kcal/mol)")
        ax.set_title(f"Round {ROUND_ID} — SHAP (top {len(top_feats)})")
        ax.grid(True, axis="x", linestyle="--", alpha=0.3)
        plt.tight_layout()
        bar_out = out_dir / f"round{ROUND_ID}_shap_bar_top{len(top_feats)}.png"
        plt.savefig(bar_out, dpi=300, bbox_inches="tight")
        plt.show()
        print(f"[SHAP] Saved bar plot → {bar_out}")

    # Beeswarm plot (top MAX_DISPLAY; colorbar LEFT, names RIGHT)
    if any(pt.lower() == "beeswarm" for pt in PLOT_TYPES) and len(top_feats) > 0:
        swarm_df = beeswarm_df.copy()

        xmin = float(swarm_df["shap_value"].min())
        xmax = float(swarm_df["shap_value"].max())
        xrng = xmax - xmin
        xpad = 0.06 * (xrng if xrng > 0 else 1.0)

        fig = plt.figure(figsize=(9.2, 6.2))
        gs = gridspec.GridSpec(1, 4, width_ratios=[0.08, 1.0, 0.06, 0.52], wspace=0.05)

        ax_cbar  = fig.add_subplot(gs[0, 0])
        ax_swarm = fig.add_subplot(gs[0, 1])
        ax_space = fig.add_subplot(gs[0, 2])
        ax_names = fig.add_subplot(gs[0, 3], sharey=ax_swarm)
        ax_space.axis("off")

        norm = Normalize(vmin=0.0, vmax=1.0)
        cmap = mpl.cm.coolwarm

        ax_swarm.scatter(
            swarm_df["shap_value"].to_numpy(),
            swarm_df["y_plot"].to_numpy(),
            c=swarm_df["feature_value_scaled"].to_numpy(),
            s=18, alpha=0.9, cmap=cmap, norm=norm, edgecolors="none"
        )
        ax_swarm.axvline(0, color="k", lw=0.9, alpha=0.6, zorder=0)

        ax_swarm.set_xlim(xmin - xpad, xmax + xpad)
        ax_swarm.set_ylim(-0.8, len(top_feats) - 1 + 0.8)
        ax_swarm.set_yticks([])
        ax_swarm.grid(True, axis="x", linestyle="--", alpha=0.25)
        ax_swarm.set_xlabel("SHAP value (impact on predicted ΔΔG‡)")
        ax_swarm.set_title(f"Round {ROUND_ID} — SHAP beeswarm (top {len(top_feats)})")

        ax_names.set_xlim(0, 1)
        ax_names.set_xticks([]); ax_names.set_yticks([])
        for spine in ax_names.spines.values():
            spine.set_visible(False)

        centers = {f: (len(top_feats)-1 - i) for i, f in enumerate(top_feats)}
        for f in top_feats:
            ax_names.text(0.0, centers[f], f, va="center", ha="left", fontsize=10)
        ax_names.set_ylim(ax_swarm.get_ylim())

        sm = ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
        cb = fig.colorbar(sm, cax=ax_cbar)
        cb.set_label("Feature value (scaled 0..1)", rotation=90)
        cb.ax.yaxis.set_label_position("left")
        cb.ax.yaxis.set_ticks_position("left")
        cb.ax.tick_params(left=True, right=False)

        fig.subplots_adjust(left=0.12, right=0.98, top=0.93, bottom=0.12, wspace=0.25)

        out_png = out_dir / f"round{ROUND_ID}_shap_beeswarm_top{len(top_feats)}.png"
        plt.savefig(out_png, dpi=300, bbox_inches="tight")
        plt.show()
        print(f"[SHAP] Saved beeswarm plot → {out_png}")

    print("\n[SHAP] Done.")

## Step 4: Select n experiments to run

In [ ]:
# === Step 4: Select next batch (HALF top-mean + HALF low-unc from top-K), record/plot Avg. EI (%) ===
# Re-suggests items previously selected but not yet measured. Excludes ONLY already measured ones.

# User settings
ROUND_ID = 0
BATCH_SIZE = 10           # total experiments to run this round
TOP_K_FOR_LOW_UNC = 10    # among top-K by mean, pick the lowest-uncertainty half

PRED_FILE = Path(f"round{ROUND_ID}_acetals_predictions.csv")  # from Step 3

# 1) Load predictions (or use in-memory df if present)
if "acetals_out" in globals():
    df_pred = acetals_out.copy()
elif "untested_out" in globals():
    df_pred = untested_out.copy()
else:
    assert PRED_FILE.exists(), f"Missing predictions for round {ROUND_ID}: {PRED_FILE}"
    df_pred = pd.read_csv(PRED_FILE)

# Basic checks / normalize
need_cols = {"Structure", "DeltaDeltaG_pred", "DeltaDeltaG_uncertainty"}
missing = need_cols - set(df_pred.columns)
assert not missing, f"Predictions missing columns: {missing}"

df_pred["Structure"] = df_pred["Structure"].astype(str).str.strip()
df_pred["DeltaDeltaG_pred"] = pd.to_numeric(df_pred["DeltaDeltaG_pred"], errors="coerce")
df_pred["DeltaDeltaG_uncertainty"] = pd.to_numeric(df_pred["DeltaDeltaG_uncertainty"], errors="coerce")
df_pred = df_pred[np.isfinite(df_pred["DeltaDeltaG_pred"])].copy()

# Keep EI if present
has_ei = "Expected_Improvement" in df_pred.columns
if has_ei:
    df_pred["Expected_Improvement"] = pd.to_numeric(df_pred["Expected_Improvement"], errors="coerce")

# 2) Exclude ONLY structures that have already been measured in prior rounds
#    (Allow re-suggesting structures selected in earlier rounds but not yet measured.)
prev_measured = set()
ledger = Path("tested_results_ledger.csv")
if ledger.exists():
    try:
        led = pd.read_csv(ledger)
        led["Structure"] = led["Structure"].astype(str).str.strip()
        # Treat only rounds strictly earlier than this one as "already measured"
        if "round" in led.columns:
            prev_measured.update(led.loc[led["round"] < ROUND_ID, "Structure"].tolist())
        else:
            prev_measured.update(led["Structure"].tolist())
    except Exception:
        pass

before = len(df_pred)
df_pred = df_pred[~df_pred["Structure"].isin(prev_measured)].copy()
dropped_prev = before - len(df_pred)
if dropped_prev > 0:
    print(f"[Round {ROUND_ID}] Excluded {dropped_prev} previously measured structures.")
if df_pred.empty:
    raise ValueError(f"[Round {ROUND_ID}] No candidates remain for selection.")

# 3) Ranking helpers
df_pred["unc_for_sort"] = np.where(np.isfinite(df_pred["DeltaDeltaG_uncertainty"]),
                                   df_pred["DeltaDeltaG_uncertainty"], np.inf)

df_sorted = df_pred.sort_values(["DeltaDeltaG_pred", "Structure"], ascending=[False, True]).reset_index(drop=True)
df_sorted["mean_rank"] = np.arange(1, len(df_sorted) + 1)
if has_ei:
    df_sorted["EI_rank"] = df_sorted["Expected_Improvement"].rank(ascending=False, method="min")

# 4) Compute split sizes (odd batch: extra goes to low-unc half)
n_total = int(BATCH_SIZE)
assert n_total >= 1, "BATCH_SIZE must be >= 1."
n_exploit = n_total // 2
n_lowunc  = n_total - n_exploit

# 5) Exploit picks: top n_exploit by mean
exploit = df_sorted.head(n_exploit).copy()
exploit["selection_role"] = "Exploit_high_mean"

# 6) Low-uncertainty picks: among top-K-by-mean (excluding exploits), take n_lowunc with smallest σ
K = min(int(TOP_K_FOR_LOW_UNC), len(df_sorted))
candidates = df_sorted.iloc[:K].copy()
candidates = candidates[~candidates["Structure"].isin(exploit["Structure"])].copy()

# Prefer finite σ when possible
cand_finite = candidates[np.isfinite(candidates["DeltaDeltaG_uncertainty"])].copy()
pool_for_lowunc = cand_finite if len(cand_finite) >= n_lowunc else candidates

low_unc = pool_for_lowunc.sort_values(
    ["unc_for_sort", "mean_rank", "Structure"], ascending=[True, True, True]
).head(n_lowunc).copy()
low_unc["selection_role"] = "High_mean_low_unc"

# Add within-top-K uncertainty rank for chosen
tmp_rank = candidates.sort_values(["unc_for_sort", "mean_rank", "Structure"], ascending=[True, True, True]) \
                     .reset_index(drop=True)
rank_map = {s: i+1 for i, s in enumerate(tmp_rank["Structure"].tolist())}
low_unc["unc_rank_topK"] = low_unc["Structure"].map(rank_map)

# 7) Merge and ensure BATCH_SIZE unique; backfill by next-best mean if short
selection = pd.concat([exploit, low_unc], ignore_index=True).drop_duplicates("Structure")
if len(selection) < n_total:
    filler = df_sorted[~df_sorted["Structure"].isin(selection["Structure"])].head(n_total - len(selection)).copy()
    filler["selection_role"] = "Fallback_high_mean"
    filler["unc_rank_topK"] = ""  # N/A
    selection = pd.concat([selection, filler], ignore_index=True)

# 8) Final tidy + save (+ audit)
for col in ["unc_rank_topK"]:
    if col not in selection.columns:
        selection[col] = ""

keep_cols = ["Structure", "DeltaDeltaG_pred", "DeltaDeltaG_uncertainty",
             "selection_role", "mean_rank", "unc_rank_topK"]
if has_ei:
    keep_cols += ["Expected_Improvement", "EI_rank"]

selection = selection[keep_cols].reset_index(drop=True)

out_path  = f"round{ROUND_ID}_batch{BATCH_SIZE}_selection.csv"
audit_out = f"round{ROUND_ID}_batch{BATCH_SIZE}_selection_audit.csv"

selection.to_csv(out_path, index=False)

audit = df_sorted.copy()
audit["is_selected"] = audit["Structure"].isin(selection["Structure"])
audit_cols = ["Structure", "DeltaDeltaG_pred", "DeltaDeltaG_uncertainty", "mean_rank", "is_selected"]
if has_ei:
    audit_cols += ["Expected_Improvement", "EI_rank"]
audit[audit_cols].to_csv(audit_out, index=False)

print(f"Round {ROUND_ID}: selected {len(selection)} candidates "
      f"({n_exploit} exploit + {n_lowunc} high-mean/low-unc; fallback if needed).")
print(f"Saved selection → {out_path}")
print(f"Saved audit     → {audit_out}")
display(selection)

# 9) Record & plot Avg. EI (%) for the SELECTED BATCH vs Round,
#    using EI defined relative to the BEST ALREADY-MEASURED POOL value ONLY.
def _prior_pool_best(round_id:int) -> float:
    """Max ΔΔG among pool items measured in prior rounds (from tested_results_ledger.csv)."""
    p = Path("tested_results_ledger.csv")
    if not p.exists():
        return float("nan")
    dfm = pd.read_csv(p)
    if "round" in dfm.columns:
        dfm = dfm[dfm["round"] < round_id]
    y = pd.to_numeric(dfm.get("DeltaDeltaG_true"), errors="coerce")
    y = y[np.isfinite(y)]
    return float(y.max()) if len(y) else float("nan")

# Ensure EI exists and is relative to prior pool best; if missing, compute it for the selected batch
if ("Expected_Improvement" not in selection.columns) or selection["Expected_Improvement"].isna().all():
    f_best_pool = _prior_pool_best(ROUND_ID)
    if np.isfinite(f_best_pool):
        mu = pd.to_numeric(selection["DeltaDeltaG_pred"], errors="coerce").to_numpy()
        sigma = pd.to_numeric(selection["DeltaDeltaG_uncertainty"], errors="coerce").to_numpy()
        eps = 1e-12
        sigma_safe = np.where(np.isfinite(sigma) & (sigma > eps), sigma, np.nan)
        z = (mu - f_best_pool) / sigma_safe
        try:
            from scipy.stats import norm
            Phi = norm.cdf(z); phi = norm.pdf(z)
        except Exception:
            import math
            erf_vec = np.vectorize(math.erf)
            Phi = 0.5 * (1.0 + erf_vec(z / np.sqrt(2.0)))
            phi = (1.0 / np.sqrt(2.0 * np.pi)) * np.exp(-0.5 * z * z)
        ei_sel = np.where(
            np.isfinite(sigma_safe),
            (mu - f_best_pool) * Phi + sigma_safe * phi,
            np.maximum(0.0, mu - f_best_pool)
        )
        selection["Expected_Improvement"] = ei_sel
        has_ei = True

# Compute batch Avg. EI (%) relative to prior pool best
if has_ei and selection["Expected_Improvement"].notna().any():
    f_best_pool = _prior_pool_best(ROUND_ID)
    if np.isfinite(f_best_pool):
        batch_avg_ei_raw = float(selection["Expected_Improvement"].mean(skipna=True))
        denom = max(1e-12, abs(f_best_pool))
        batch_avg_ei_pct = 100.0 * batch_avg_ei_raw / denom

        # Persist history
        hist_path = Path("ei_history_batch.csv")
        if hist_path.exists():
            hist = pd.read_csv(hist_path)
        else:
            hist = pd.DataFrame(columns=["round", "avg_ei_raw", "avg_ei_pct", "best_pool_y_at_round"])

        # Idempotent upsert
        hist = hist[hist["round"] != ROUND_ID]
        hist = pd.concat([
            hist,
            pd.DataFrame([{
                "round": ROUND_ID,
                "avg_ei_raw": batch_avg_ei_raw,
                "avg_ei_pct": batch_avg_ei_pct,
                "best_pool_y_at_round": f_best_pool,
            }])
        ], ignore_index=True).sort_values("round")
        hist.to_csv(hist_path, index=False)

        print(f"\n[Round {ROUND_ID}] Batch Avg. EI (raw, vs prior measured pool): {batch_avg_ei_raw:.6g}")
        print(f"[Round {ROUND_ID}] Batch Avg. EI (% of pool best={f_best_pool:.4g}): {batch_avg_ei_pct:.3f}%")
        print(f"Saved/updated EI history → {hist_path}")

        # Plot Avg. EI (%) vs. Round — integer ticks
        plt.figure(figsize=(5.5, 3.6))
        plt.plot(hist["round"], hist["avg_ei_pct"], marker="o")
        ax = plt.gca()
        r_max = int(hist["round"].max()) if len(hist) else ROUND_ID
        ticks = np.arange(0, r_max + 1, 1)
        ax.set_xticks(ticks)
        ax.set_xlim(-0.1, r_max + 0.1)
        ax.set_xlabel("Round")
        ax.set_ylabel(f"Avg. EI (%) — batch of {BATCH_SIZE} (vs prior pool best)")
        ax.set_title("Average Expected Improvement (%) by Round")
        ax.grid(True, linestyle="--", alpha=0.3)
        plt.tight_layout()
        plt.show()
    else:
        print("\n[Info] No prior measured pool yet (e.g., Round 0) — skipping EI history update & plot.")
else:
    print("\n[Info] Expected_Improvement not available for selection — skipping EI history update & plot.")


## Step 5: compare predicted vs. measured ∆∆G values

In [ ]:
# === Step 5: Add ALL measured results for this round, evaluate, and update ledgers ===
# - Includes measured structures NOT present in this round's selection file
# - Evaluates predicted vs measured for anything we can match to a prediction
# - Saves: eval CSV, metrics CSV, round ledger, cumulative ledger
# - Plots: parity (±1σ), MAE vs Round, Avg. AI (%) vs Round, Max ΔΔG‡ per round

# ---------------------------
# User inputs (edit these)
# ---------------------------
ROUND_ID = 0

# Either supply lists...
TESTED_STRUCTURES = []
MEASURED_DDG = []  # kcal/mol

# If you know the batch size/file name from Step 4, set it here; otherwise we auto-detect.
BATCH_SIZE = None  # e.g. 6; if None, auto-detect latest selection file for this round

# ---------------------------
# Imports (uses Step 1's environment)
# ---------------------------
from pathlib import Path
import re, math, io, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score

# ---------------------------
# 0) Build measured dataframe from user inputs
# ---------------------------
if 'MEASURED_RESULTS' in globals() and isinstance(MEASURED_RESULTS, dict):
    measured_df = pd.DataFrame(
        {"Structure": list(MEASURED_RESULTS.keys()),
         "DeltaDeltaG_true": list(MEASURED_RESULTS.values())}
    )
else:
    assert len(TESTED_STRUCTURES) == len(MEASURED_DDG), "TESTED_STRUCTURES and MEASURED_DDG lengths differ."
    measured_df = pd.DataFrame({"Structure": TESTED_STRUCTURES, "DeltaDeltaG_true": MEASURED_DDG})

measured_df["Structure"] = measured_df["Structure"].astype(str).str.strip()
measured_df["DeltaDeltaG_true"] = pd.to_numeric(measured_df["DeltaDeltaG_true"], errors="coerce")

# ---------------------------
# 1) Locate this round's selection file (from Step 4)
# ---------------------------
def _find_latest_selection_for_round(r: int) -> Path:
    matches = sorted(
        Path(".").glob(f"round{r}_batch*_selection.csv"),
        key=lambda p: p.stat().st_mtime, reverse=True
    )
    return matches[0] if matches else None

sel_path = None
if BATCH_SIZE is not None:
    cand = Path(f"round{ROUND_ID}_batch{BATCH_SIZE}_selection.csv")
    if cand.exists():
        sel_path = cand
if sel_path is None:
    sel_path = _find_latest_selection_for_round(ROUND_ID)

assert sel_path is not None and sel_path.exists(), \
    f"No selection file found for round {ROUND_ID} (pattern: round{ROUND_ID}_batch*_selection.csv)."

m = re.match(rf"round{ROUND_ID}_batch(\d+)_selection\.csv$", sel_path.name)
if m:
    BATCH_SIZE = int(m.group(1))

print(f"Using selection file: {sel_path.name}  (BATCH_SIZE={BATCH_SIZE})")
sel_df = pd.read_csv(sel_path)
sel_df["Structure"] = sel_df["Structure"].astype(str).str.strip()

# ---------------------------
# 2) Load this round's full predictions (to cover items not in selection)
# ---------------------------
def _detect_mean_std_cols(df):
    cols = df.columns.tolist()
    preferred = [
        ("DeltaDeltaG_predicted_mean", "DeltaDeltaG_predicted_variance"),
        ("DeltaDeltaG_mean", "DeltaDeltaG_std"),
        ("DeltaDeltaG_mu", "DeltaDeltaG_sigma"),
        ("posterior_mean_DeltaDeltaG", "posterior_std_DeltaDeltaG"),
        ("DeltaDeltaG_pred", "DeltaDeltaG_uncertainty"),
        ("mean_0", "std_0"), ("mu_0","sigma_0"),
        ("pred_mean","pred_std"), ("mean","std"),
    ]
    for m, s in preferred:
        if m in cols and s in cols:
            return m, s
    mean_like = [c for c in cols if re.search(r"(mean|mu|pred)", c, re.I)]
    std_like  = [c for c in cols if re.search(r"(std|sigma|uncert|var)", c, re.I)]
    if mean_like and std_like:
        return mean_like[0], std_like[0]
    return None, None

pred_full_path = Path(f"round{ROUND_ID}_predictions_full.csv")
if not pred_full_path.exists():
    # Fallbacks: the exact pred_* file, or most recent pred_* if multiple
    alt = Path(f"pred_edbo_gpr_round{ROUND_ID}.csv")
    if alt.exists():
        pred_full_path = alt
    else:
        alts = list(Path(".").glob("pred_*round*.csv")) + list(Path(".").glob("pred_*.csv"))
        alts = sorted(alts, key=lambda p: p.stat().st_mtime, reverse=True)
        pred_full_path = alts[0] if alts else None

pred_full = None
mean_col = unc_col = None
if pred_full_path is not None and pred_full_path.exists():
    pred_full = pd.read_csv(pred_full_path)
    if "Structure" in pred_full.columns:
        pred_full["Structure"] = pred_full["Structure"].astype(str).str.strip()
        mean_col, unc_col = _detect_mean_std_cols(pred_full)
        if mean_col is None or unc_col is None:
            print("[Warning] Could not auto-detect mean/std columns in predictions; metrics may be limited.")
    else:
        print("[Warning] Full predictions file does not have 'Structure' column; metrics will only include selected items.")

# ---------------------------
# 3) Merge truth with selection (to annotate selection_role), then extend with non-selected measurements
# ---------------------------
sel_annot = sel_df.merge(measured_df, on="Structure", how="left")
sel_missing = sel_annot["DeltaDeltaG_true"].isna().sum()
if sel_missing > 0:
    print(f"[Info] {sel_missing} of {len(sel_df)} selected structures have no measurement provided yet (this round).")

extra_measured = set(measured_df["Structure"]) - set(sel_df["Structure"])
if extra_measured:
    print(f"[Note] {len(extra_measured)} measured structure(s) were NOT in this round's selection: {sorted(extra_measured)}")

# Build evaluation frame that includes ALL measured this round
eval_df = pd.concat([
    sel_annot,
    measured_df[~measured_df["Structure"].isin(sel_df["Structure"])].assign(
        DeltaDeltaG_pred=np.nan,
        DeltaDeltaG_uncertainty=np.nan,
        selection_role="(not_selected_this_round)",
        mean_rank=np.nan,
        unc_rank_topK=""
    )
], ignore_index=True)

# If we have a predictions table, fill in predictions for ANY measured structure we can find there
if pred_full is not None and "Structure" in pred_full.columns and mean_col is not None and unc_col is not None:
    # Convert variance->std if needed
    pred_full = pred_full.copy()
    if any(k in str(unc_col).lower() for k in ["var", "variance"]):
        pred_full["__std__"] = np.sqrt(np.clip(pd.to_numeric(pred_full[unc_col], errors="coerce").to_numpy(), 0.0, None))
        unc_name = "__std__"
    else:
        unc_name = unc_col
    # Left-join for any rows still missing predictions
    still_missing_mask = eval_df["DeltaDeltaG_pred"].isna()
    if still_missing_mask.any():
        to_fill = eval_df.loc[still_missing_mask, ["Structure"]].merge(
            pred_full[["Structure", mean_col, unc_name]],
            on="Structure", how="left"
        ).rename(columns={mean_col: "DeltaDeltaG_pred", unc_name: "DeltaDeltaG_uncertainty"})
        eval_df.loc[still_missing_mask, ["DeltaDeltaG_pred", "DeltaDeltaG_uncertainty"]] = to_fill[
            ["DeltaDeltaG_pred", "DeltaDeltaG_uncertainty"]
        ].values

# ---------------------------
# 4) Compute metrics on available pairs (ALL measured this round)
# ---------------------------
y_true = pd.to_numeric(eval_df.get("DeltaDeltaG_true"), errors="coerce").to_numpy()
y_pred = pd.to_numeric(eval_df.get("DeltaDeltaG_pred"),  errors="coerce").to_numpy()
y_unc  = pd.to_numeric(eval_df.get("DeltaDeltaG_uncertainty", np.nan), errors="coerce").to_numpy()

mask = np.isfinite(y_true) & np.isfinite(y_pred)
y_true_m = y_true[mask]; y_pred_m = y_pred[mask]
y_unc_m  = y_unc[mask] if (y_unc is not None and y_unc.size == len(eval_df)) else np.full_like(y_pred_m, np.nan)

if len(y_true_m) > 0:
    mse  = mean_squared_error(y_true_m, y_pred_m)
    rmse = float(np.sqrt(mse))
    mae  = float(np.mean(np.abs(y_true_m - y_pred_m)))
    r2   = r2_score(y_true_m, y_pred_m) if len(np.unique(y_true_m)) > 1 else np.nan
else:
    mse = rmse = mae = r2 = np.nan

n_sel = len(sel_df)
n_meas = int(pd.notna(eval_df["DeltaDeltaG_true"]).sum())
n_extra = len(set(measured_df["Structure"]) - set(sel_df["Structure"]))

print(f"Round {ROUND_ID} — evaluation on measured {n_meas} structure(s)")
print(f" - Of these, in selection this round: {n_meas - n_extra}")
print(f" - Extra measured not in selection  : {n_extra}")
print(f"R²   : {r2: .4f}" if np.isfinite(r2) else "R²   :  n/a")
print(f"RMSE : {rmse: .4f} kcal/mol" if np.isfinite(rmse) else "RMSE :  n/a")
print(f"MAE  : {mae: .4f}  kcal/mol" if np.isfinite(mae)  else "MAE  :  n/a")

# ---------------------------
# 5) Save artifacts (with residuals) — includes ALL measured this round
# ---------------------------
eval_df["DeltaDeltaG_true"] = pd.to_numeric(eval_df.get("DeltaDeltaG_true"), errors="coerce")
eval_df["residual"]  = eval_df["DeltaDeltaG_true"] - pd.to_numeric(eval_df.get("DeltaDeltaG_pred"), errors="coerce")
eval_df["abs_error"] = eval_df["residual"].abs()
eval_df["in_selection_this_round"] = eval_df["Structure"].isin(sel_df["Structure"])

eval_out    = f"round{ROUND_ID}_batch{BATCH_SIZE}_selected_with_truth.csv"
metrics_out = f"round{ROUND_ID}_batch{BATCH_SIZE}_metrics.csv"
eval_df.to_csv(eval_out, index=False)
pd.DataFrame({"metric": ["R2","RMSE","MAE"], "value": [r2, rmse, mae]}).to_csv(metrics_out, index=False)
print(f"Saved: {eval_out}, {metrics_out}")
display(eval_df)

# ---------------------------
# 6) Parity plot WITH uncertainty (±1σ) for all evaluated pairs
#     - Colors: keep selection roles where present; others marked as '(not_selected_this_round)'
# ---------------------------
if mask.sum() > 0:
    role_colors = {
        "Exploit_high_mean":   "#1f77b4",  # blue
        "High_mean_low_unc":   "#2ca02c",  # green
        "Fallback_high_mean":  "#ff7f0e",  # orange
        "(not_selected_this_round)": "#7f7f7f",  # gray
    }

    plot_df = eval_df.loc[mask].copy()
    plot_df["selection_role"] = plot_df.get("selection_role", "(not_selected_this_round)").fillna("(not_selected_this_round)").astype(str)
    plot_df["DeltaDeltaG_uncertainty"] = pd.to_numeric(plot_df.get("DeltaDeltaG_uncertainty", np.nan), errors="coerce")

    fig, ax = plt.subplots(figsize=(5.6, 5.6))
    handles, labels = [], []

    # deterministic order for legend
    for role in ["Exploit_high_mean","High_mean_low_unc","Fallback_high_mean","(not_selected_this_round)"]:
        sub = plot_df[plot_df["selection_role"] == role]
        if sub.empty:
            continue
        color = role_colors.get(role, "#7f7f7f")
        pts = ax.scatter(sub["DeltaDeltaG_true"], sub["DeltaDeltaG_pred"], label=role, s=40, alpha=0.95, color=color, zorder=3)
        if np.isfinite(sub["DeltaDeltaG_uncertainty"]).any():
            ax.errorbar(sub["DeltaDeltaG_true"], sub["DeltaDeltaG_pred"],
                        yerr=sub["DeltaDeltaG_uncertainty"], fmt="none",
                        elinewidth=1, capsize=3, alpha=0.8, color=color, zorder=2)
        handles.append(pts); labels.append(role)

    # y=x diagonal spanning the plotted range (including error bars)
    mins = np.nanmin(np.concatenate([
        plot_df["DeltaDeltaG_true"].to_numpy(),
        (plot_df["DeltaDeltaG_pred"] - plot_df["DeltaDeltaG_uncertainty"].fillna(0)).to_numpy()
    ]))
    maxs = np.nanmax(np.concatenate([
        plot_df["DeltaDeltaG_true"].to_numpy(),
        (plot_df["DeltaDeltaG_pred"] + plot_df["DeltaDeltaG_uncertainty"].fillna(0)).to_numpy()
    ]))
    pad  = 0.05 * (maxs - mins if maxs > mins else 1.0)
    line_min, line_max = mins - pad, maxs + pad
    ax.plot([line_min, line_max], [line_min, line_max], linewidth=1, color="black", alpha=0.7)

    ax.set_xlim(line_min, line_max)
    ax.set_ylim(line_min, line_max)
    ax.set_xlabel("Measured ΔΔG‡ (kcal/mol)")
    ax.set_ylabel("Predicted ΔΔG‡ (kcal/mol)")
    ttl = f"Round {ROUND_ID} — Parity Plot with σ (n={len(plot_df)})"
    if np.isfinite(mae):
        ttl += f"\nRMSE={rmse:.3f} kcal/mol, MAE={mae:.3f}"
    ax.set_title(ttl)
    ax.set_aspect('equal', adjustable='box')
    ax.grid(True, linestyle="--", alpha=0.3)
    ax.legend(handles, labels, title="Selection role", frameon=True, fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    print("[Info] Skipping parity plot: no (prediction, measurement) pairs yet.")

# ---------------------------
# 7) MAE vs Round — robust to ROUND_ID=0 and missing files
# ---------------------------
def _find_round_file(pattern_fmt: str, r: int) -> Path:
    pats = list(Path(".").glob(pattern_fmt.format(r=r)))
    if not pats:
        return None
    return sorted(pats, key=lambda p: p.stat().st_mtime, reverse=True)[0]

def _read_round_metrics(r: int):
    mpath = _find_round_file("round{r}_batch*_metrics.csv", r)
    npath = _find_round_file("round{r}_batch*_selected_with_truth.csv", r)
    mae_val = rmse_val = r2_val = np.nan
    n_val = np.nan
    if mpath is not None and mpath.exists():
        m = pd.read_csv(mpath)
        def _get(name):
            s = m.loc[m["metric"].str.upper() == name.upper(), "value"]
            return float(s.iloc[0]) if not s.empty else np.nan
        r2_val   = _get("R2")
        rmse_val = _get("RMSE")
        mae_val  = _get("MAE")
    if npath is not None and npath.exists():
        ev = pd.read_csv(npath)
        yt = pd.to_numeric(ev.get("DeltaDeltaG_true"), errors="coerce")
        yp = pd.to_numeric(ev.get("DeltaDeltaG_pred"), errors="coerce")
        n_val = int((yt.notna() & yp.notna()).sum())
    return mae_val, rmse_val, r2_val, n_val

rows = []
for r in range(0, ROUND_ID + 1):
    mae_r, rmse_r, r2_r, n_r = _read_round_metrics(r)
    if r == ROUND_ID:  # prefer in-memory numbers we just computed
        mae_r, rmse_r, r2_r, n_r = mae, rmse, r2, int(len(y_true_m))
    # Only record rounds that have at least one finite metric
    if np.isfinite(mae_r) or np.isfinite(rmse_r) or np.isfinite(r2_r):
        rows.append({"round": r, "mae": mae_r, "rmse": rmse_r, "r2": r2_r, "n": n_r})

if rows:
    mae_log = pd.DataFrame(rows).sort_values("round")
    mae_log_path = Path("mae_history_batch.csv")
    mae_log.to_csv(mae_log_path, index=False)

    if mae_log["mae"].notna().any():
        plt.figure(figsize=(5.6, 3.8))
        plt.plot(mae_log["round"], mae_log["mae"], marker="o")
        ax = plt.gca()
        r_max = int(mae_log["round"].max())
        ax.set_xticks(np.arange(0, r_max + 1, 1))
        ax.set_xlim(-0.1, r_max + 0.1)
        ax.set_xlabel("Round")
        ax.set_ylabel("MAE (kcal/mol)")
        ax.set_title("Batch MAE by Round")
        ax.grid(True, linestyle="--", alpha=0.3)
        plt.tight_layout()
        plt.show()
else:
    print("[Info] MAE history: nothing to plot yet (no finite metrics).")

# ---------------------------
# 8) Avg. Actual Improvement (AI %) vs Round — robust to ROUND_ID=0
#     Uses ALL measured this round (as saved above)
# ---------------------------
def _round_truth_df(r: int) -> pd.DataFrame:
    p = _find_round_file("round{r}_batch*_selected_with_truth.csv", r)
    if p is None or not p.exists():
        return pd.DataFrame()
    df = pd.read_csv(p)
    df["DeltaDeltaG_true"] = pd.to_numeric(df.get("DeltaDeltaG_true"), errors="coerce")
    return df

ai_rows = []
best_prev = np.nan
for r in range(0, ROUND_ID + 1):
    df_r = _round_truth_df(r)
    if df_r.empty:
        continue
    y_true_r = df_r["DeltaDeltaG_true"]
    if r == 0:
        # Seed prior-best with round 0 measurements; AI is undefined for r=0
        if y_true_r.notna().any():
            cand_max = float(y_true_r.max())
            best_prev = cand_max if not np.isfinite(best_prev) else max(best_prev, cand_max)
        continue

    if np.isfinite(best_prev) and y_true_r.notna().any():
        yv = y_true_r.dropna().to_numpy()
        imp = np.maximum(0.0, yv - best_prev)
        avg_ai_raw = float(np.mean(imp)) if imp.size else np.nan
        denom = max(1e-12, abs(best_prev))
        avg_ai_pct = 100.0 * avg_ai_raw / denom
        ai_rows.append({"round": r, "avg_ai_raw": avg_ai_raw, "avg_ai_pct": avg_ai_pct, "best_prev_y": best_prev})
        best_prev = max(best_prev, float(np.max(yv)))
    else:
        ai_rows.append({"round": r, "avg_ai_raw": np.nan, "avg_ai_pct": np.nan, "best_prev_y": best_prev})

if ai_rows:
    ai_hist = pd.DataFrame(ai_rows).sort_values("round")
    ai_hist_path = Path("ai_history_batch.csv")
    ai_hist.to_csv(ai_hist_path, index=False)

    if ai_hist["avg_ai_pct"].notna().any():
        plt.figure(figsize=(5.6, 3.8))
        plt.plot(ai_hist["round"], ai_hist["avg_ai_pct"], marker="o")
        ax = plt.gca()
        r_max = int(max(ai_hist["round"].max(), 0))
        ax.set_xticks(np.arange(0, r_max + 1, 1))
        ax.set_xlim(-0.1, r_max + 0.1)
        ax.set_xlabel("Round")
        ax.set_ylabel("Avg. AI (%) — batch")
        ax.set_title("Average Actual Improvement (%) by Round")
        ax.grid(True, linestyle="--", alpha=0.3)
        plt.tight_layout()
        plt.show()

        row_cur = ai_hist.loc[ai_hist["round"] == ROUND_ID]
        if not row_cur.empty and np.isfinite(row_cur["avg_ai_pct"].iloc[0]):
            print(f"[Round {ROUND_ID}] Batch Avg. AI (raw): {row_cur['avg_ai_raw'].iloc[0]:.6g} kcal/mol")
            print(f"[Round {ROUND_ID}] Batch Avg. AI (% of prior best={row_cur['best_prev_y'].iloc[0]:.4g}): "
                  f"{row_cur['avg_ai_pct'].iloc[0]:.3f}%")
else:
    print("[Info] Avg. AI history: nothing to plot yet (need ≥1 prior round and measured truths).")

# ---------------------------
# 9) Max measured ΔΔG‡ per round — plot round max and running best
# ---------------------------
def _round_max_ddg(r: int) -> float:
    p = _find_round_file("round{r}_batch*_selected_with_truth.csv", r)
    if p is not None and p.exists():
        df_r = pd.read_csv(p)
        vals = pd.to_numeric(df_r.get("DeltaDeltaG_true"), errors="coerce")
        if vals.notna().any():
            return float(vals.max())
    return np.nan

rows_max = []
for r in range(0, ROUND_ID + 1):
    mval = _round_max_ddg(r)
    if np.isfinite(mval):
        rows_max.append({"round": r, "round_max_ddg": mval})

if rows_max:
    maxlog = pd.DataFrame(rows_max).sort_values("round").reset_index(drop=True)
    maxlog["best_so_far"] = maxlog["round_max_ddg"].cummax()
    maxlog.to_csv("max_ddg_by_round.csv", index=False)

    plt.figure(figsize=(5.6, 3.8))
    plt.plot(maxlog["round"], maxlog["round_max_ddg"], marker="o", label="Round max ΔΔG‡")
    plt.plot(maxlog["round"], maxlog["best_so_far"], linestyle="--", label="Best-so-far")
    ax = plt.gca()
    r_max = int(maxlog["round"].max())
    ax.set_xticks(np.arange(0, r_max + 1, 1))
    ax.set_xlim(-0.1, r_max + 0.1)
    ax.set_xlabel("Round")
    ax.set_ylabel("Max ΔΔG‡ in batch (kcal/mol)")
    ax.set_title("Max measured ΔΔG‡ per round (and best-so-far)")
    ax.grid(True, linestyle="--", alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("[Note] No valid ΔΔG‡ measurements found to plot per-round maxima.")

# ---------------------------
# 10) Record the tested reactions for NEXT round (round + cumulative ledgers)
# ---------------------------
# Per-round ledger (used by Step 1 to highlight “new” bars in the histogram)
round_ledger = measured_df.copy()
round_ledger.insert(0, "round", ROUND_ID)
round_ledger_out = f"round{ROUND_ID}_tested_results.csv"
round_ledger.to_csv(round_ledger_out, index=False)
print(f"Saved round tested-results ledger → {round_ledger_out}")

# Cumulative ledger (used by Step 1 to assemble df_train across rounds)
ledger_path = Path("tested_results_ledger.csv")
if ledger_path.exists():
    cum = pd.read_csv(ledger_path)
    cum.columns = [str(c).strip() for c in cum.columns]
    # Normalize required cols
    if "Structure" not in cum.columns:
        raise ValueError("Existing tested_results_ledger.csv has no 'Structure' column.")
    if "DeltaDeltaG_true" not in cum.columns:
        if "DeltaDeltaG" in cum.columns:
            cum = cum.rename(columns={"DeltaDeltaG": "DeltaDeltaG_true"})
        else:
            cum["DeltaDeltaG_true"] = np.nan
    if "round" not in cum.columns:
        cum["round"] = np.nan
else:
    cum = pd.DataFrame(columns=["round", "Structure", "DeltaDeltaG_true"])

# Upsert by Structure: latest measurement overwrites earlier entry
cum["Structure"] = cum["Structure"].astype(str).str.strip()
mask_old = cum["Structure"].isin(round_ledger["Structure"])
cum = cum[~mask_old]
cum = pd.concat([cum, round_ledger], ignore_index=True)

# Sort if possible
sort_cols = [c for c in ["round", "Structure"] if c in cum.columns]
if sort_cols:
    cum = cum.sort_values(sort_cols).reset_index(drop=True)

cum.to_csv(ledger_path, index=False)
print(f"Updated cumulative tested-results ledger → {ledger_path.name}")